# EDA do AlertaRio

Inspecao dos dados pluviometricos ja convertidos para Parquet. A conversao
de arquivos brutos deve ser feita por scripts versionados, nao por este notebook.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists():
            return candidate
    raise RuntimeError('Nao foi possivel localizar a raiz do repositorio.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DEFAULT_PARQUET_ROOT = (
    PROJECT_ROOT / 'data' / 'datasets' / 'raw' / 'alertario' / 'pluviometricos_parquet'
)
PARQUET_ROOT = Path(os.environ.get('ALERTARIO_PARQUET_ROOT', DEFAULT_PARQUET_ROOT))

print(f'Projeto: {PROJECT_ROOT}')
print(f'Parquets: {PARQUET_ROOT}')
if not PARQUET_ROOT.is_dir():
    raise FileNotFoundError(
        'Pasta de Parquets nao encontrada. Defina ALERTARIO_PARQUET_ROOT ou '
        'prepare os dados com o script correspondente.'
    )

## Arquivos e esquema

O notebook espera arquivos no formato `*.parquet`, com as colunas
`estacao_id`, `estacao`, `dia_utc` e `m15`.

In [ ]:
files = sorted(PARQUET_ROOT.glob('*.parquet'))
print(f'Arquivos encontrados: {len(files)}')
if not files:
    raise FileNotFoundError(f'Nenhum Parquet encontrado em {PARQUET_ROOT}.')

sample = pd.read_parquet(files[0])
print(f'Arquivo de exemplo: {files[0].name}')
print(f'Shape: {sample.shape}')
display(sample.head())

In [ ]:
required_columns = {'estacao_id', 'estacao', 'dia_utc', 'm15'}
missing = required_columns - set(sample.columns)
if missing:
    raise ValueError(f'Colunas obrigatorias ausentes: {sorted(missing)}')

print('Valores nulos em m15:', int(sample['m15'].isna().sum()))
print('Valores negativos em m15:', int((sample['m15'] < 0).sum()))
print('Valor maximo de m15:', sample['m15'].max())

## Cobertura temporal por estacao

In [ ]:
coverage_frames = []
for index, path in enumerate(files, start=1):
    frame = pd.read_parquet(path, columns=['estacao_id', 'estacao', 'dia_utc'])
    frame['dia_utc'] = pd.to_datetime(frame['dia_utc'], utc=True, errors='coerce')
    frame = frame.dropna(subset=['estacao_id', 'dia_utc'])
    frame['ano'] = frame['dia_utc'].dt.year
    coverage_frames.append(frame[['estacao_id', 'estacao', 'ano']].drop_duplicates())
    if index % 50 == 0 or index == len(files):
        print(f'Arquivos analisados: {index}/{len(files)}')

coverage = pd.concat(coverage_frames, ignore_index=True).drop_duplicates()
years_by_station = (
    coverage.groupby(['estacao_id', 'estacao'])['ano']
    .agg(lambda years: sorted(years.unique()))
    .reset_index(name='anos_disponiveis')
    .sort_values('estacao_id')
)
stations_by_year = (
    coverage.groupby('ano')['estacao_id']
    .nunique()
    .reset_index(name='quantidade_estacoes')
    .sort_values('ano')
)

print(f'Estacoes observadas: {len(years_by_station)}')
display(stations_by_year)
display(years_by_station)

In [ ]:
required_years = set(range(2012, 2025))
complete_stations = years_by_station[
    years_by_station['anos_disponiveis'].map(lambda years: required_years.issubset(years))
].copy()

print(f'Estacoes com cobertura completa de 2012 a 2024: {len(complete_stations)}')
display(complete_stations)

## Proximos passos

- Use `notebooks/02_geospatial/01_mapa_estacoes_alertario.ipynb` para a
  distribuicao espacial das estacoes.
- Use `scripts/analysis/exportar_maximos_websirene.py` para exportar os
  maiores acumulados do WebSirene.
- Mantenha a construcao dos memmaps e targets nos scripts do pipeline.